In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_path = "../data/"


df_airports = pd.read_csv(data_path + "airportgroups.csv")
print(f"Airport Groups: {df_airports.shape}")

df_hist = pd.read_csv(data_path + "historical_flights.csv")
print(f"Historical Flights: {df_hist.shape}")

df_sched = pd.read_csv(data_path + "schedule_oct2025.csv")
print(f"Schedule Oct 2025: {df_sched.shape}")



print("\nDataset Summary:")
datasets = {
    'Airport Groups': df_airports,
    'Historical Flights': df_hist,
    'Schedule Oct 2025': df_sched,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Airport Groups: (21, 2)
Historical Flights: (410442, 11)
Schedule Oct 2025: (3984, 8)

Dataset Summary:
Airport Groups: 21 rows × 2 columns
Historical Flights: 410,442 rows × 11 columns
Schedule Oct 2025: 3,984 rows × 8 columns
Historical Flights: (410442, 11)
Schedule Oct 2025: (3984, 8)

Dataset Summary:
Airport Groups: 21 rows × 2 columns
Historical Flights: 410,442 rows × 11 columns
Schedule Oct 2025: 3,984 rows × 8 columns


In [14]:
# ====================================================================================================
# TEMPORAL FEATURE ENGINEERING
# ====================================================================================================

def create_temporal_features(df, datetime_col='std'):
    """Create comprehensive temporal features from datetime column"""
    print(f"Creating temporal features from {datetime_col}...")
    
    # Parse datetime
    df[f'{datetime_col}_parsed'] = pd.to_datetime(df[datetime_col])
    dt = df[f'{datetime_col}_parsed'].dt
    
    # Basic temporal features
    df['hour'] = dt.hour
    df['day_of_week'] = dt.dayofweek
    df['day_name'] = dt.day_name()
    df['month'] = dt.month
    df['day_of_month'] = dt.day
    df['week_of_year'] = dt.isocalendar().week
    df['quarter'] = dt.quarter
    df['year'] = dt.year
    
    # Advanced temporal features
    df['is_weekend'] = (dt.dayofweek >= 5).astype(int)
    df['is_monday'] = (dt.dayofweek == 0).astype(int)
    df['is_friday'] = (dt.dayofweek == 4).astype(int)
    
    # Hour categories
    df['hour_category'] = pd.cut(df['hour'], 
                                bins=[0, 6, 9, 12, 17, 20, 24], 
                                labels=['Night', 'Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Late_Evening'],
                                include_lowest=True)
    
    # Rush hour indicators
    df['morning_rush'] = ((df['hour'] >= 5) & (df['hour'] <= 7)).astype(int)
    df['evening_rush'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)
    df['business_hours'] = ((df['hour'] >= 7) & (df['hour'] <= 18)).astype(int)
    
    # TODO: Holiday indicators (requires holiday calendar)
    
    
    
    # Cyclical features (important for capturing circular nature of time)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    # Season mapping
    season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
                  3: 'Spring', 4: 'Spring', 5: 'Spring',
                  6: 'Summer', 7: 'Summer', 8: 'Summer',
                  9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}
    df['season'] = df['month'].map(season_map)
    
    print(f"Created {sum('hour' in col or 'day' in col or 'month' in col or 'season' in col or 'rush' in col for col in df.columns)} temporal features")
    return df

# Apply temporal features to historical data
print("STEP 1: TEMPORAL FEATURE ENGINEERING")
print("=" * 50)
df_hist_enhanced = df_hist.copy()
df_hist_enhanced = create_temporal_features(df_hist_enhanced, 'std')

# Show temporal features created
temporal_cols = [col for col in df_hist_enhanced.columns if any(x in col.lower() for x in ['hour', 'day', 'month', 'season', 'rush', 'weekend', 'business'])]
print(f"Temporal features created: {temporal_cols}")

STEP 1: TEMPORAL FEATURE ENGINEERING
Creating temporal features from std...
Created 18 temporal features
Temporal features created: ['hour', 'day_of_week', 'day_name', 'month', 'day_of_month', 'is_weekend', 'is_monday', 'is_friday', 'hour_category', 'morning_rush', 'evening_rush', 'business_hours', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'season']
Created 18 temporal features
Temporal features created: ['hour', 'day_of_week', 'day_name', 'month', 'day_of_month', 'is_weekend', 'is_monday', 'is_friday', 'hour_category', 'morning_rush', 'evening_rush', 'business_hours', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'season']


In [ ]:
# ====================================================================================================
# FLIGHT DENSITY & OPERATIONAL FEATURES
# ====================================================================================================

def create_density_features(df):
    """Create flight density and operational load features"""
    print("Creating flight density features...")
    
    # Sort by datetime for window operations
    df = df.sort_values('std_parsed').reset_index(drop=True)
    
    # Airport group flight counts by time windows
    df['date'] = df['std_parsed'].dt.date
    df['hour_slot'] = df['std_parsed'].dt.floor('H')
    
    # Flights per hour per airport group
    hourly_counts = df.groupby(['dep_airport_group', 'hour_slot']).size().reset_index()
    hourly_counts.columns = ['dep_airport_group', 'hour_slot', 'flights_this_hour']
    df = df.merge(hourly_counts, on=['dep_airport_group', 'hour_slot'], how='left')
    
    # Flights per day per airport group
    daily_counts = df.groupby(['dep_airport_group', 'date']).size().reset_index()
    daily_counts.columns = ['dep_airport_group', 'date', 'flights_this_day']
    df = df.merge(daily_counts, on=['dep_airport_group', 'date'], how='left')
    
    # System-wide load
    system_hourly = df.groupby('hour_slot').size().reset_index()
    system_hourly.columns = ['hour_slot', 'system_flights_this_hour']
    df = df.merge(system_hourly, on='hour_slot', how='left')
    
    # Relative load (airport group vs system)
    df['relative_load'] = df['flights_this_hour'] / (df['system_flights_this_hour'] + 1)
    
    # Service type diversity
    service_counts = df.groupby(['dep_airport_group', 'hour_slot', 'service_type']).size().unstack(fill_value=0)
    service_counts.columns = [f'service_{col}_count' for col in service_counts.columns]
    service_counts = service_counts.reset_index()
    df = df.merge(service_counts, on=['dep_airport_group', 'hour_slot'], how='left')
    
    # Fill NaN values
    service_cols = [col for col in df.columns if col.startswith('service_') and col.endswith('_count')]
    df[service_cols] = df[service_cols].fillna('J')
    
    print(f"Created density features: flights_this_hour, flights_this_day, system_flights_this_hour, relative_load")
    print(f"Created service type features: {len(service_cols)} service type counts")
    
    return df

# Apply density features
print("\nSTEP 2: FLIGHT DENSITY FEATURES")
print("=" * 50)
df_hist_enhanced = create_density_features(df_hist_enhanced)


STEP 2: FLIGHT DENSITY FEATURES
Creating flight density features...
Created density features: flights_this_hour, flights_this_day, system_flights_this_hour, relative_load
Created service type features: 16 service type counts
Created density features: flights_this_hour, flights_this_day, system_flights_this_hour, relative_load
Created service type features: 16 service type counts


In [16]:
# ====================================================================================================
# DELAY & OPERATIONAL RELIABILITY FEATURES
# ====================================================================================================

def create_operational_features(df):
    """Create operational reliability and delay-based features"""
    print("Creating operational reliability features...")
    
    # Calculate delays if actual times exist
    if 'atd' in df.columns and pd.notna(df['atd']).any():
        df['atd_parsed'] = pd.to_datetime(df['atd'])
        df['departure_delay'] = (df['atd_parsed'] - df['std_parsed']).dt.total_seconds() / 60  # minutes
        
        # Delay categories
        df['delay_category'] = pd.cut(df['departure_delay'], 
                                    bins=[-np.inf, -5, 15, 60, np.inf],
                                    labels=['Early', 'On_Time', 'Delayed', 'Severely_Delayed'])
        
        # On-time performance by airport group
        df['is_on_time'] = (df['departure_delay'].abs() <= 15).astype(int)
        
    # Cancellation rates
    if 'cancelled' in df.columns:
        df['is_cancelled'] = df['cancelled'].fillna(0).astype(int)
    else:
        df['is_cancelled'] = 0
    
    # Flight duration (scheduled)
    if 'sta' in df.columns:
        df['sta_parsed'] = pd.to_datetime(df['sta'])
        df['scheduled_duration'] = (df['sta_parsed'] - df['std_parsed']).dt.total_seconds() / 3600  # hours
        
        # Duration categories
        df['duration_category'] = pd.cut(df['scheduled_duration'],
                                       bins=[0, 1, 2, 4, 8, np.inf],
                                       labels=['Short', 'Medium', 'Long', 'Very_Long', 'Ultra_Long'])
    
    print("✓ Created operational features: delays, on-time performance, cancellations, flight duration")
    return df

# Apply operational features
print("\nSTEP 3: OPERATIONAL RELIABILITY FEATURES")
print("=" * 50)
df_hist_enhanced = create_operational_features(df_hist_enhanced)


STEP 3: OPERATIONAL RELIABILITY FEATURES
Creating operational reliability features...
✓ Created operational features: delays, on-time performance, cancellations, flight duration
✓ Created operational features: delays, on-time performance, cancellations, flight duration
